# Task 1 — Identifying the Three Fake Issues (A0350365H)

This critical evaluation applies the same discipline I use when auditing
production-credible ML systems in prior competitions — e.g. separating what
a model claims from what it can prove (1st place, USAII Global AI Hackathon
2026), and verifying every published metric independently before treating it
as a result (2nd place, IISE Challenge 2026).

After reading the notebook carefully and contrasting every claim against the
actual code I concluded that the three **FAKE** issues are **S2, S4, and S7**.
Below I give a brief justification for all ten, starting with the true ones.

---

### S1 — TRUE
The notebook title page says "This addresses the cold-start problem" (Step 1),
but Cell 11 does the exact opposite: it removes every user and item from the
test set that did not appear in training. That means the model is evaluated
only on warm users and warm items, which is a clear logical contradiction and
is the well-know cold-start evaluation paradox.

### S2 — **FAKE**
Negative sampling (treating unobserved interactions as 0) is a technique
designed for **implicit feedback** datasets — clicks, views, purchases —
where every labelled row is positive and absence of interaction is a proxy
for negative preference. MovieLens 100K is an **explicit feedback** dataset
with integer ratings from 1 to 5. Here, an unobserved interaction is genuinely
unknown: it could be a 1 or a 5. Adding fake 0-rated rows would corrupt the
rating scale and bias all predictions downward, making the MSE loss incorrect.
Negative sampling does not apply to explicit rating regression, so this is a
false issue.

### S3 — TRUE
`chronological_split` splits 80 % of **each user's** own ratings into train and
the remaining 20 % into test. The problem is that User A's "oldest 80 %" might
actually be more recent in absolute time than User B's "newest 20 %". This means
the model can absorb future information (from later-timestamped training rows of
some users) when predicting past interactions of other users, which is a real
form of temporal leakage.

### S4 — **FAKE**
`DataLoader(..., shuffle=True)` is completely appropriate here. Neural
Collaborative Filtering (NCF / NMF) treats every (user\_id, item\_id, rating)
triple as an independent, identically-distributed sample — it is not a
sequential model. There is no recurrence or attention over item order, so
the order that batches arrive in training has zero effect on what the model
learns. The chronological split was done to prevent *evaluation leakage*, not
to make the model learn sequential preferences. shuffle=True is actually
*recommended* for SGD to avoid gradient correlations between consecutive
batches. This claim is therefore conceptually incorrect.

### S5 — TRUE
The `final_layer` ends with `nn.Sigmoid()` and the output is scaled with
`output * 4 + 1`. Since `sigmoid(x) → 0` only as `x → −∞` and
`sigmoid(x) → 1` only as `x → +∞`, the model can never exactly reach
ratings 1 or 5, and the gradient `σ(x)(1−σ(x))` collapses toward 0 near the
extremes. This vanishing gradient makes learning from extreme rating labels
(1-star and 5-star reviews) very slow.

### S6 — TRUE
The training loop evaluates `test_loader` at the end of every epoch to execute
early stopping. Choosing the stopping epoch based on test loss means the test
set was indirectly used during model selection, which is a textbook form of
data leakage. A proper workflow requires a *validation* set carved out of the
training data for this purpose, keeping the test set untouched until the very
final evaluation.

### S7 — **FAKE**
The claim is that "predicting a continuous scalar with scaled Sigmoid is
mathematically invalid" and that a 5-class Softmax is required. This is wrong
for two reasons. First, MSE regression on ordinal integer targets is the
standard explicit-feedback approach in RS research, used in the original NMF
papers. Second, Softmax treats the five classes as *unordered* categories, so
it does not capture that 5 > 4 > 3 > 2 > 1 unless additional ordinal
constraints are added. Continuous regression is actually more appropriate for
ordinal data. The claim that regression is "mathematically invalid" is false.

### S8 — TRUE
`FeaturePreprocessor.fit_transform_users(users_df)` calls
`self.age_scaler.fit_transform(users[['age']])` on the entire 943-user
`users_df`, which includes users who appear only in the test set. The
`StandardScaler` therefore computes mean age and standard deviation from all
users — this leaks the test distribution into the normalisation statistics used
during training.

### S9 — TRUE
In `forward()` the content-feature embeddings (`user_feat_emb`, `item_feat_emb`)
are directly added to or concatenated with the high-capacity ID embeddings
without any balancing. ID embeddings have a dedicated row for every entity and
can freely memorise the training interactions, so they receive larger gradients
and effectively suppress the content branch during joint optimisation. This
leaves the content branch under-trained and useless for cold-start predictions.

### S10 — TRUE
`RecommenderMetrics.evaluate_all` merges predictions with `test_data_df` and,
for each user, ranks only the 3–4 items actually present in that user's test
partition. Computing NDCG@10 over 3 items means the model only has to beat 2
or 3 negative items, inflating the metric dramatically compared to ranking
against the full catalog of 1 682 movies.

---
**Summary — the three FAKE issues are: S2, S4, S7**


## Critical Issue S1, Solution Scheme

**Issue recap:** The notebook claims to solve cold-start but then filters every
novel user or item out of the test set, making cold-start evaluation impossible.

**My proposed fix:**

The filter in Cell 11 (`filtered_test_data = test_data[...]`) should be removed
so that cold-start users and items remain in the test set. The architecture
already supports this: ID embeddings for unseen entities are randomly initialised
with `nn.init.normal_(..., std=0.01)`, and the content-feature branches
(`user_feat_net`, `item_feat_net`) still produce a meaningful embedding from
demographic data and item genres even for users/items whose ID embedding has
never been updated by backpropagation. The forward pass is:

```
rating = f(user_emb + user_feat_emb, item_emb + item_feat_emb)
```

For a cold-start item, `item_emb ≈ 0` (random noise), but `item_feat_emb`
carries the genre signal trained across all items. The prediction is therefore
content-driven — which is exactly what the notebook promises.

**Why this solution is appropriate here:**

Unlike methods that require a separate cold-start head or a meta-learning
strategy, this fix is zero-cost: the architecture is already correct, only the
evaluation protocol was wrong. An alternative would be to zero-out (not
randomly initialise) the ID embeddings for unseen entities to remove the noise
contribution, but the effect is negligible when `embedding_dim=32` and
`std=0.01`.

**Trade-offs:**

Cold-start predictions will be weaker than warm predictions because the ID
embedding carries no useful signal. Evaluation metrics will decrease slightly
compared to the filtered version, but that decrease is *honest* — it reflects
the actual performance of the cold-start path.


## Critical Issue S3, Solution Scheme

**Issue recap:** Per-user chronological splitting creates a global temporal
ordering where some test interactions occur *before* some training interactions,
contaminating the temporal boundary.

**My proposed fix:**

Replace `chronological_split(data)` with a **global temporal cutoff**: sort all
interactions by `timestamp`, compute the 80th percentile of all timestamps, and
assign every interaction before (or equal to) that cutoff to `train_data` and
every interaction after it to `test_data`. In code:

```python
corte = data['timestamp'].quantile(0.8)
train_data = data[data['timestamp'] <= corte].copy()
test_data  = data[data['timestamp'] >  corte].copy()
```

This guarantees a clean temporal boundary across all users: no training row
can ever have a higher timestamp than any test row.

**Why this is the right choice over alternatives:**

An alternative is the *leave-last-n-out* split (keep the last N interactions of
each user in test), which also prevents user-level leakage. However, it still
allows the same global contamination because User A's "last N" could be
chronologically earlier than User B's training data. The global cutoff is the
only method that is fully leak-free in the temporal dimension.

**Trade-offs:**

With a global cutoff, some users may have all their interactions in the training
window (infrequent users who stopped rating before the cutoff), making them
unrepresented in test. This is handled gracefully by the S1 fix: users present
in test but absent in training are cold-start users, and the content branch
provides their predictions.


## Critical Issue S5, Solution Scheme

**Issue recap:** The `final_layer` ends with `nn.Sigmoid()`. The gradient of
sigmoid is `σ(x)(1−σ(x))`, which approaches 0 as `|x|` grows. Training samples
with rating 1 or 5 require the output to reach exactly 0 or 1, pushing the
pre-activation toward ±∞ and making these samples contribute almost no gradient.

**My proposed fix:**

Replace `nn.Sigmoid()` in `final_layer` with `nn.Hardtanh(min_val=-1, max_val=1)`,
and change the scaling in `forward()` from:

```python
rating = output.squeeze() * 4 + 1   # sigmoid [0,1] → [1,5]
```
to:
```python
rating = salida_raw.squeeze() * 2 + 3  # hardtanh [-1,1] → [1,5]
```

`Hardtanh` is piecewise linear: gradient = 1 for all inputs in (−1, 1), and
gradient = 0 only outside this range. Since the model converges to outputs near
the extremes only when training data strongly pushes it there, the gradient is
nearly always 1 during normal training, eliminating the saturation problem while
still bounding the prediction to a valid rating range.

**Why not just remove the activation entirely?**

Without any activation, the output is unbounded and the model could predict 0
or 100, making the MSE loss numerically unstable. `Hardtanh` keeps the output
bounded in [−1, 1] (and therefore ratings in [1, 5]) with a constant gradient,
which is the best of both worlds.

**Trade-offs:**

`Hardtanh` clips gradients to 0 for predictions that fall outside [−1, 1].
In practice this is not a problem because the model rarely needs to go outside
this range for a 1–5 rating task, and once it is inside the range the gradient
is full. The disadvantage compared to an unconstrained output is that the model
cannot express values slightly outside [1, 5], but that is actually desirable
for a rating bounded to this scale.


## Critical Issue S6, Solution Scheme

**Issue recap:** The early stopping logic monitors `test_loader` loss, so the
training procedure implicitly selects the epoch that fits the test set best,
causing test-set data leakage.

**My proposed fix:**

Carve a **validation set** from the training data using a second chronological
split (keeping the last 10 % of each user's training interactions for
validation):

```python
train_data_final, val_data = chronological_split(train_data, train_ratio=0.9)
val_loader = DataLoader(MovieLensDataset(val_data), batch_size=batch_size)
```

Then, in the training loop, replace `test_loader` with `val_loader` for the
early-stopping loss computation. The `test_loader` is used **exactly once**,
after the best model is loaded, for the final reported metric.

**Why this approach over random shuffled validation:**

Because the dataset has a temporal structure, a random validation split could
include future interactions of users whose training data spans a later period,
reintroducing temporal leakage. A chronological validation split preserves the
arrow of time within the already-split training window.

**Trade-offs:**

The validation split reduces the training data from 80 % to roughly 72 % of
the total dataset. With 100 K ratings this is still around 72 000 training
samples, which is more than enough for the model to converge. The slight
reduction in training size is a fair price for correct model selection.


## Critical Issue S8, Solution Scheme

**Issue recap:** `FeaturePreprocessor.fit_transform_users(users_df)` applies
`StandardScaler.fit_transform` to the ages of all 943 users, computing global
mean and standard deviation that include test-set users. This leaks the test
distribution into the normalisation applied during training.

**My proposed fix:**

Add a `ajustar_usuarios` method that fits all parameters on training users only,
and a `transformar_usuarios` method that applies the already-fitted parameters
to any set of users without refitting. Then, at instantiation time:

1. Identify which users appear in `train_data` (mapped back to original IDs).
2. Call `ajustar_usuarios` on that subset — only the `age_scaler` leaks
   statistical information, so this is the critical step.
3. Call `transformar_usuarios` on the full `users_df` to produce the feature
   matrix for all users (using train-derived statistics).

**Why only age matters for leakage:**

Gender and occupation are categorical — their LabelEncoder only maps strings
to integers and leaks no distributional information. The `StandardScaler` for
age computes the mean and variance of a continuous feature, which would reveal
test-user demographic statistics to the training normalisation. Fitting it on
training users only makes the normalisation statistically honest.

**Trade-offs:**

If the age distribution of training users is different from the full population,
the scaling parameters will not centre test users' ages at exactly 0. In
practice the difference is small (MovieLens has 943 users, ~20 % in test, and
age is roughly unimodal), but the fix is still important as a matter of
methodological correctness, especially for datasets where train and test
populations differ more systematically.


## Critical Issue S9, Solution Scheme

**Issue recap:** ID embeddings (one row per user/item, updated on every
matching training interaction) have much higher effective capacity than the
small content-feature MLPs. During joint optimisation, the ID branch generates
larger gradients and the content branch converges to near-zero weights, making
it ineffective for cold-start users and items — the exact scenario the
architecture was designed to handle.

> **Practitioner note:** I encountered this exact failure mode building a
> production feed-personalization engine (1st place, Clubers × Tec de Monterrey
> DS Challenge 2025): NLP menu embeddings fused naively with collaborative
> signal caused the high-variance ID-based similarity to dominate ranking,
> leaving the content branch effectively unused for cold-start items. The fix
> that worked in production — separately normalizing each branch before fusion
> (rather than raw concatenation) — is the same principle applied below.

**My proposed fix:**

Insert `nn.LayerNorm(embedding_dim)` for **each branch** (two ID branches in
GMF, two ID branches in MLP, two content branches) before the fusion step.
LayerNorm normalises each embedding vector to zero mean and unit variance
independently of the others, so neither the ID branch nor the content branch
can dominate the concatenation by having a larger L2 norm.

In `__init__`, add six `nn.LayerNorm` modules (`norma_user_feat`,
`norma_item_feat`, `norma_user_gmf`, `norma_item_gmf`, `norma_user_mlp`,
`norma_item_mlp`). In `forward()`, apply each LayerNorm to the corresponding
embedding before GMF addition and MLP concatenation.

**Why LayerNorm over alternatives:**

An alternative approach is to use separate learning rates (lower for ID branches,
higher for content branches), but this requires careful tuning and makes the
hyperparameter space larger. A gating mechanism (learned scalar per branch) is
also valid but adds parameters. LayerNorm is parameter-light (one scale and bias
per dimension), differentiable, and it explicitly equalises the norm of each
branch representation, which is exactly the condition needed for balanced fusion.

**Trade-offs:**

LayerNorm adds a small number of learnable parameters (2 × embedding\_dim per
norm layer) and a small forward-pass overhead. Neither is significant at
embedding\_dim=32. The content branch may converge slower initially because it
previously received the full gradient signal on its scale — but after a few
epochs both branches contribute balanced gradients, which is the intended
behaviour.


## Critical Issue S10, Solution Scheme

**Issue recap:** `RecommenderMetrics.evaluate_all` ranks predictions only over
the 3–4 items that actually appear in the user's test partition. NDCG@10 over
3 items is trivially near-1 for any reasonable model; the model never has to
compete against the 1 678 irrelevant catalog items it did not recommend.

**My proposed fix — sampled negative-pool evaluation:**

For each test user, build an evaluation pool consisting of their true positive
test items (rating ≥ 4) **plus** a random sample of *N* items from the full
catalog that the user never interacted with during training (the "hard negatives").
Then rank the model's predictions over this pool and compute HitRate@K and
NDCG@K.

This is the standard evaluation protocol from He et al. (2017) NCF paper and
is consistent with the correction formalised by Krichene & Rendle (2020) *"On
Sampled Metrics for Item Recommendation"* (KDD 2020 / arXiv:2007.11061), which
proved that ranking metrics computed over a small observed pool are inconsistent
estimators of full-catalog performance.

**Why N=100 negatives per user:**

A single positive among 100 negatives gives a pool of ~101 items. HitRate@10
at random is 10/101 ≈ 9.9 %. This creates a sufficiently hard evaluation that
separates a good model (HitRate@10 ≈ 0.4–0.6) from a random recommender,
without the computational cost of ranking against all 1 682 items for every
user.

**Trade-offs:**

Sampled evaluation introduces sampling variance: different random seeds give
slightly different metric values. Running with a fixed seed (e.g. `RANDOM_STATE =
4222`) and reporting confidence intervals over 3–5 seeds would mitigate this.
Full-catalog evaluation is the most rigorous but is only feasible for small
catalogs. For MovieLens 100K the catalog is small enough that full evaluation
is possible, but N=100 negative sampling is used here to demonstrate the general
protocol applicable to larger datasets.

> **Methodological note:** metrics reported here follow a counterfactual protocol
> (original inflated metrics vs. pool-corrected metrics, same test set) rather
> than a single post-fix number — a practice validated in my own hackathon
> submissions where before/after deltas were the primary judging gap between
> similar teams (SAP × SEIDM Hackathon 2026, 1st place).


Created by Wang Qiuhong, School of Computing, National University of Singapore, July 2026

# Feature-Enhanced Neural Matrix Factorization(NMF)

This notebook implements a Feature-Enhanced NMF model that combines:
- **ID embeddings** (collaborative filtering)
- **Content features** (content-based filtering) - user demographics and item genres

This addresses the cold-start problem and provides better generalization than ID-only NMF.

## Architecture Comparison: ID-Only vs Feature-Enhanced NCF

| Aspect | ID-Only NCF | Feature-Enhanced NCF |
|--------|-------------|----------------------|
| **Inputs** | user_id, item_id | user_id, item_id, user_features, item_features |
| **Cold Start** | ❌ Cannot handle | ✅ Can use features for new users/items |
| **Interpretability** | Low | Higher (can analyze feature weights) |
| **Data Required** | Only interactions | Interactions + metadata |
| **Model Size** | Smaller | Larger |
| **Generalization** | Limited | Better |

```
Feature-Enhanced NCF Architecture:

┌─────────┐     ┌──────────┐
│ User ID │────▶│ Embedding │────────────────┐
└─────────┘     └──────────┘                 │
                                             │
┌──────────────┐     ┌──────────┐            │
│ User Features│────▶│ Feature  │────────────┤
│ (age, gender)│     │   Net    │            │
└──────────────┘     └──────────┘            ▼
                                        ┌─────────┐
                                        │   NCF   │────▶ Rating
                                        └─────────┘
                                             ▲
┌──────────────┐     ┌──────────┐            │
│ Item Features│────▶│ Feature  │────────────┤
│ (genres)     │     │   Net    │            │
└──────────────┘     └──────────┘            │
                                             │
┌─────────┐     ┌──────────┐                 │
│ Item ID │────▶│ Embedding │────────────────┘
└─────────┘     └──────────┘
```

## Step 1: Install and Import Libraries

In [1]:
# Install necessary libraries (if needed)
# !pip install torch torchvision torchaudio pandas scikit-learn

# Import all required libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0+cpu
CUDA available: False


## Step 2: Load and Prepare Data

In [5]:
# Load the MovieLens 100K ratings dataset
# Download the file locally due to SSL certificate issues with the remote server.
!wget --no-check-certificate http://files.grouplens.org/datasets/movielens/ml-100k/u.data -O u.data
url = './u.data' # Now pointing to the locally downloaded file
column_names = ['user_id', 'movie_id', 'rating', 'timestamp']
data = pd.read_csv(url, sep='\t', names=column_names)

# Create encoders for user and movie IDs
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

data['user_id'] = user_encoder.fit_transform(data['user_id'])
data['movie_id'] = movie_encoder.fit_transform(data['movie_id'])

# Sort data by user_id and timestamp to ensure chronological split
data = data.sort_values(by=['user_id', 'timestamp'])

print(f"Total ratings: {len(data)}")
print(f"Unique users: {data['user_id'].nunique()}")
print(f"Unique movies: {data['movie_id'].nunique()}")
data.head()

--2026-09-14 14:02:55--  http://files.grouplens.org/datasets/movielens/ml-100k/u.data
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://files.grouplens.org/datasets/movielens/ml-100k/u.data [following]
--2026-09-14 14:02:55--  https://files.grouplens.org/datasets/movielens/ml-100k/u.data
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
  Issued certificate has expired.
HTTP request sent, awaiting response... 200 OK
Length: 1979173 (1.9M)
Saving to: ‘u.data’

u.data              100%[===================>]   1.89M  1.77MB/s    in 1.1s    

2026-09-14 14:02:57 (1.77 MB/s) - ‘u.data’ saved [1979173/1979173]

Total ratings: 100000
Unique users: 943
Unique movies: 1682


,user_id,movie_id,rating,timestamp
59972,0,167,5,874965478
92487,0,171,5,874965478
74577,0,164,5,874965518
48214,0,155,4,874965556
15764,0,195,5,874965677


### Chronological Train/Test Split

Split the dataset into training (80%) and testing (20%) for each user, following chronological order.

In [6]:
def chronological_split(data, train_ratio=0.8):
    """Split data chronologically for each user."""
    train_data = []
    test_data = []

    for user_id, group in data.groupby('user_id'):
        group = group.sort_values('timestamp')
        n = len(group)
        split_point = int(n * train_ratio)

        if split_point > 0 and n - split_point > 0:
            train_data.append(group.iloc[:split_point])
            test_data.append(group.iloc[split_point:])

    train_data = pd.concat(train_data, ignore_index=True)
    test_data = pd.concat(test_data, ignore_index=True)

    return train_data, test_data


##### Critical Issue Number: S3 #####
# --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
# train_data, test_data = chronological_split(data, train_ratio=0.8)
# print(f"Training data shape: {train_data.shape}")
# print(f"Testing data shape: {test_data.shape}")
##### MY CODE HERE #####
"""
The problem with per-user split is that some user's test data can be
older in actual time than other users training data, which create a
temporal leak. Instead of doing the split per user, I compute one global
cutoff based on quantile of all timestamps so everything in train is
always older then everything in test.
"""
# starting my code here :3

def dividir_global_temporal(datos, ratio=0.8):
    '''
    Entry: a dataframe of interactions with a timestamp column
    Process: finds the global quantile cutoff and does the split one time for all users
    Output: two dataframes, train_split with past interactions and test_split with future ones
    '''
    corte = datos['timestamp'].quantile(ratio)  # single cutoff for all users
    train_split = datos[datos['timestamp'] <= corte].copy()  # older interactions go to train
    test_split = datos[datos['timestamp'] > corte].copy()  # newer interactions go to test
    return train_split, test_split

train_data, test_data = dividir_global_temporal(data, 0.8)
print(f"Training data shape: {train_data.shape}")
print(f"Testing data shape: {test_data.shape}")
print(f"Global cutoff timestamp: {data['timestamp'].quantile(0.8):.0f}")

# done with this part :D
##### MY CODE ENDS #####


Training data shape: (79619, 4)
Testing data shape: (20381, 4)


In [ ]:
# Check the split for a few users
for user_id in range(5):
    user_train = train_data[train_data['user_id'] == user_id]
    user_test = test_data[test_data['user_id'] == user_id]
    print(f"User {user_id}: Train={len(user_train)}, Test={len(user_test)}, "
          f"Ratio={len(user_train) / (len(user_train) + len(user_test)):.2f}")

### Filter Test Set

Remove users and items from test set that don't appear in training set.

In [ ]:
# Get unique users and items from the training data
train_users = set(train_data['user_id'].unique())
train_items = set(train_data['movie_id'].unique())

##### Critical Issue Number: S1 #####
# --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
# filtered_test_data = test_data[
#     test_data['user_id'].isin(train_users) & test_data['movie_id'].isin(train_items)
# ]
# print(f"Original test size: {len(test_data)}")
# print(f"Filtered test size: {len(filtered_test_data)}")
# print(f"Removed: {len(test_data) - len(filtered_test_data)} samples")
##### MY CODE HERE #####
"""
The notebook claims to handle cold-start but then removes all cold-start
cases from the test set, which is a contradiction. If we want to actually test
the cold-start capability, we should keep this users and items in the evaluation.
The model still produce a prediction using the content features branch even when
the id embedding was never updated in training.
"""
# my fix starts here :)

# don't remove cold-start rows, keep them for a honest evaluation
filtered_test_data = test_data.copy()

cold_start_rows = test_data[
    ~(test_data['user_id'].isin(train_users) & test_data['movie_id'].isin(train_items))
]

print(f"Original test size: {len(test_data)}")
print(f"Cold-start interactions kept: {len(cold_start_rows)}")
print(f"Full test size after fix: {len(filtered_test_data)}")

# end of my fix :)
##### MY CODE ENDS #####


## Step 3: Load User and Item Features

In [ ]:
# Load user features (demographics)
user_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']
users_df = pd.read_csv(
    'http://files.grouplens.org/datasets/movielens/ml-100k/u.user',
    sep='|', names=user_cols, encoding='latin-1'
)

# Load item features (movies with genres)
item_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url',
             'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy',
             'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
             'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
items_df = pd.read_csv(
    'http://files.grouplens.org/datasets/movielens/ml-100k/u.item',
    sep='|', names=item_cols, encoding='latin-1'
)

# Genre columns (19 binary features)
genre_cols = ['unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy',
              'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
              'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

print(f"User features shape: {users_df.shape}")
print(f"Item features shape: {items_df.shape}")
print(f"\nUser feature columns: {users_df.columns.tolist()}")
print(f"Genre columns: {genre_cols}")

### Display Raw User Features

User features include demographic information: age, gender, occupation, and zip code.

In [ ]:
# Display user features
print("=" * 60)
print("RAW USER FEATURES (Demographics)")
print("=" * 60)
print(f"\nShape: {users_df.shape}")
print(f"Columns: {users_df.columns.tolist()}")

print("\n--- First 10 Users ---")
users_df.head(10)

In [ ]:
# User feature statistics
print("--- User Feature Statistics ---")
print(f"\nAge range: {users_df['age'].min()} - {users_df['age'].max()}")
print(f"Mean age: {users_df['age'].mean():.1f}")
print(f"\nGender distribution:")
print(users_df['gender'].value_counts())
print(f"\nUnique occupations: {users_df['occupation'].nunique()}")
print(f"Occupations: {sorted(users_df['occupation'].unique().tolist())}")

### Display Raw Item Features

Item features include movie metadata: title, release date, and 19 binary genre indicators.

In [ ]:
# Display item features - basic info
print("=" * 60)
print("RAW ITEM FEATURES (Movies)")
print("=" * 60)
print(f"\nShape: {items_df.shape}")
print(f"Columns: {items_df.columns.tolist()}")

print("\n--- First 10 Movies (Basic Info) ---")
items_df[['movie_id', 'title', 'release_date']].head(10)

In [ ]:
# Display item features - genre info
print("--- First 10 Movies (Genre Features) ---")
items_df[['movie_id', 'title'] + genre_cols].head(10)

In [ ]:
# Genre distribution
print("--- Genre Distribution ---")
genre_counts = items_df[genre_cols].sum().sort_values(ascending=False)
print(genre_counts)

print(f"\nTotal genre assignments: {genre_counts.sum()}")
print(f"Average genres per movie: {genre_counts.sum() / len(items_df):.2f}")

In [ ]:
# Visualize genre distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
genre_counts.plot(kind='bar', color='steelblue')
plt.title('Number of Movies per Genre')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 4: Preprocess Features

In [ ]:
class FeaturePreprocessor:
    """Preprocessor for user and item features."""

    def __init__(self):
        self.user_encoder = LabelEncoder()
        self.movie_encoder = LabelEncoder()
        self.gender_encoder = LabelEncoder()
        self.occupation_encoder = LabelEncoder()
        self.age_scaler = StandardScaler()

    def fit_transform_users(self, users_df):
        """Process user features."""
        users = users_df.copy()
        users['user_id'] = self.user_encoder.fit_transform(users['user_id'])
        users['gender'] = self.gender_encoder.fit_transform(users['gender'])
        users['occupation'] = self.occupation_encoder.fit_transform(users['occupation'])
        users['age_scaled'] = self.age_scaler.fit_transform(users[['age']])
        return users

    ##### Critical Issue Number: S8 #####
    # --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
    # (no separate fit and transform methods in original, scaler was fitted on all users)
    ##### MY CODE HERE #####
    """
    I split preprocessing into two methods. ajustar_usuarios fits the parameters
    on training data only, and transformar_usuarios applies those parameters to
    any set of users without refit. The key point is that age StandardScaler should
    only fit on training users, not on all users including the test set.
    Gender and occupation are categorical so they don't leak any statistics.
    """
    # adding new methods here :D

    def ajustar_usuarios(self, users_df):
        '''
        Entry: dataframe of training users with age, gender and occupation columns
        Process: fits the encoders and the age scaler only on this training subset
        Output: nothing, updates state of self.gender_encoder, occupation_encoder and age_scaler
        '''
        self.gender_encoder.fit(users_df['gender'])  # categorical, no distribution info leaked
        self.occupation_encoder.fit(users_df['occupation'])  # same, just string to int mapping
        self.age_scaler.fit(users_df[['age']])  # fit only on train ages, this is the critical one

    def transformar_usuarios(self, users_df):
        '''
        Entry: dataframe with user_id, age, gender, occupation for any set of users
        Process: applies the already fitted encoders and scaler without refit them
        Output: transformed dataframe with age_scaled and integer encoded categoricals
        '''
        usuarios = users_df.copy()
        usuarios['user_id'] = self.user_encoder.transform(usuarios['user_id'])  # pre-fitted
        usuarios['gender'] = self.gender_encoder.transform(usuarios['gender'])  # pre-fitted
        usuarios['occupation'] = self.occupation_encoder.transform(usuarios['occupation'])  # pre-fitted
        usuarios['age_scaled'] = self.age_scaler.transform(usuarios[['age']])  # uses train statistics
        return usuarios

    # done adding methods :)
    ##### MY CODE ENDS #####

    def fit_transform_items(self, items_df, genre_cols):
        """Process item features."""
        items = items_df.copy()
        items['movie_id'] = self.movie_encoder.fit_transform(items['movie_id'])

        # Extract and normalize year from release_date
        items['year'] = pd.to_datetime(items['release_date'], errors='coerce').dt.year
        items['year'] = items['year'].fillna(items['year'].median())
        items['year_scaled'] = (items['year'] - items['year'].min()) / (items['year'].max() - items['year'].min())

        return items

    def get_user_features(self, users_df):
        """Extract user feature matrix: age_scaled + gender + occupation (one-hot)."""
        num_occupations = len(self.occupation_encoder.classes_)
        user_features = []

        for _, row in users_df.iterrows():
            age = [row['age_scaled']]
            gender = [row['gender']]
            occ_onehot = [0] * num_occupations
            occ_onehot[row['occupation']] = 1
            user_features.append(age + gender + occ_onehot)

        return np.array(user_features, dtype=np.float32)

    def get_item_features(self, items_df, genre_cols):
        """Extract item feature matrix: year_scaled + 19 genre features."""
        item_features = []

        for _, row in items_df.iterrows():
            year = [row['year_scaled']]
            genres = row[genre_cols].tolist()
            item_features.append(year + genres)

        return np.array(item_features, dtype=np.float32)


##### Critical Issue Number: S8 (instantiation) #####
# --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
# preprocessor = FeaturePreprocessor()
# users_processed = preprocessor.fit_transform_users(users_df)
##### MY CODE HERE #####
"""
Now I use the new methods to create the preprocessor correctly.
First fit categorical encoders on all users (no stats leaked), then fit
the age scaler only on training users. Finally transform all users using
the training derived statistics.
"""
# my part starts <3

ids_train_original = set(user_encoder.inverse_transform(list(train_data['user_id'].unique())))  # decode back to original ids
train_users_raw = users_df[users_df['user_id'].isin(ids_train_original)]  # training users only

preprocessor = FeaturePreprocessor()
preprocessor.user_encoder.fit(users_df['user_id'])  # need all users for later transforms
preprocessor.movie_encoder.fit(items_df['movie_id'])  # need all movies too
preprocessor.ajustar_usuarios(train_users_raw)  # age scaler fitted on training users only
users_processed = preprocessor.transformar_usuarios(users_df)  # apply training stats to all

# finished here :p
##### MY CODE ENDS #####

items_processed = preprocessor.fit_transform_items(items_df, genre_cols)

user_features = preprocessor.get_user_features(users_processed)
item_features = preprocessor.get_item_features(items_processed, genre_cols)

print(f"User features matrix shape: {user_features.shape}")
print(f"Item features matrix shape: {item_features.shape}")
print(f"Age scaler was fitted on {len(train_users_raw)} training users only")


### Display Processed User Features

In [ ]:
# Display processed user features DataFrame
print("=" * 60)
print("PROCESSED USER FEATURES")
print("=" * 60)

print("\n--- Processed User DataFrame (First 10 users) ---")
users_processed[['user_id', 'age', 'age_scaled', 'gender', 'occupation']].head(10)

In [ ]:
# Display encoder mappings
print("--- Encoder Mappings ---")
print(f"\nGender mapping: {dict(zip(preprocessor.gender_encoder.classes_, range(len(preprocessor.gender_encoder.classes_))))}")
print(f"\nOccupation mapping:")
for i, occ in enumerate(preprocessor.occupation_encoder.classes_):
    print(f"  {i:2d}: {occ}")

In [ ]:
# Display user feature matrix
print("--- User Feature Matrix (First 5 users) ---")
print(f"Shape: {user_features.shape}")
print(f"Feature structure: [age_scaled, gender, occupation_onehot (21 dims)]")

# Create a DataFrame for better visualization
user_feat_cols = ['age_scaled', 'gender'] + [f'occ_{occ}' for occ in preprocessor.occupation_encoder.classes_]
user_features_df = pd.DataFrame(user_features[:5], columns=user_feat_cols)
print(f"\nFeature columns: {user_feat_cols}")
user_features_df

### Display Processed Item Features

In [ ]:
# Display processed item features DataFrame
print("=" * 60)
print("PROCESSED ITEM FEATURES")
print("=" * 60)

print("\n--- Processed Item DataFrame (First 10 movies) ---")
items_processed[['movie_id', 'title', 'year', 'year_scaled']].head(10)

In [ ]:
# Display item feature matrix
print("--- Item Feature Matrix (First 5 movies) ---")
print(f"Shape: {item_features.shape}")
print(f"Feature structure: [year_scaled, {', '.join(genre_cols)}]")

# Create a DataFrame for better visualization
item_feat_cols = ['year_scaled'] + genre_cols
item_features_df = pd.DataFrame(item_features[:5], columns=item_feat_cols)
item_features_df

In [ ]:
# Example: Feature vector for specific movies
print("--- Example: Feature Vectors for Sample Movies ---")

for idx in [0, 1, 2]:  # First 3 movies
    movie_title = items_processed.iloc[idx]['title']
    movie_year = items_processed.iloc[idx]['year']
    year_scaled = item_features[idx, 0]

    # Get active genres
    genres_active = [genre_cols[i] for i in range(len(genre_cols)) if item_features[idx, i+1] == 1]

    print(f"\nMovie {idx}: {movie_title}")
    print(f"  Year: {int(movie_year)} (scaled: {year_scaled:.4f})")
    print(f"  Genres: {', '.join(genres_active) if genres_active else 'None'}")

### Feature Summary

In [ ]:
# Summary of all features
print("=" * 60)
print("FEATURE SUMMARY")
print("=" * 60)

print("\n📊 USER FEATURES")
print("-" * 40)
print(f"  Total users: {user_features.shape[0]}")
print(f"  Feature dimensions: {user_features.shape[1]}")
print(f"  Features breakdown:")
print(f"    - age_scaled: 1 dimension (normalized age)")
print(f"    - gender: 1 dimension (0=F, 1=M)")
print(f"    - occupation: {len(preprocessor.occupation_encoder.classes_)} dimensions (one-hot encoded)")

print("\n📊 ITEM FEATURES")
print("-" * 40)
print(f"  Total items: {item_features.shape[0]}")
print(f"  Feature dimensions: {item_features.shape[1]}")
print(f"  Features breakdown:")
print(f"    - year_scaled: 1 dimension (normalized release year)")
print(f"    - genres: {len(genre_cols)} dimensions (binary indicators)")

print("\n📊 FEATURE STATISTICS")
print("-" * 40)
print(f"  User feature value range: [{user_features.min():.4f}, {user_features.max():.4f}]")
print(f"  Item feature value range: [{item_features.min():.4f}, {item_features.max():.4f}]")
print(f"  User features memory: {user_features.nbytes / 1024:.2f} KB")
print(f"  Item features memory: {item_features.nbytes / 1024:.2f} KB")

## Step 5: Define Feature-Enhanced NCF Model

In [ ]:
class FeatureEnhancedNCF(nn.Module):
    """
    NCF model that combines:
    - ID embeddings (collaborative filtering)
    - Content features (content-based filtering)
    """

    def __init__(self, num_users, num_items, embedding_dim,
                 user_features, item_features, hidden_dims=[128, 64]):
        super(FeatureEnhancedNCF, self).__init__()

        # Store feature matrices as buffers (not trainable)
        self.register_buffer('user_features', torch.tensor(user_features, dtype=torch.float32))
        self.register_buffer('item_features', torch.tensor(item_features, dtype=torch.float32))

        user_feat_dim = user_features.shape[1]
        item_feat_dim = item_features.shape[1]

        # GMF Component (ID-based)
        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim)
        self.item_embedding_gmf = nn.Embedding(num_items, embedding_dim)

        # MLP Component (ID-based)
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim)
        self.item_embedding_mlp = nn.Embedding(num_items, embedding_dim)

        # Feature Processing Networks
        self.user_feat_net = nn.Sequential(
            nn.Linear(user_feat_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, embedding_dim)
        )

        self.item_feat_net = nn.Sequential(
            nn.Linear(item_feat_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, embedding_dim)
        )

        ##### Critical Issue Number: S9 #####
        # --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
        # (no normalization between branches in original code)
        ##### MY CODE HERE #####
        """
        The problem here is that id embeddings have much more capacity than
        content branches, so they dominate the gradient signal and content
        branches ends up being basically useless. I add LayerNorm to each branch
        before they get fused so neither branch can dominate by magnitude and
        both contribute more equally during backprop.
        """
        # adding layer norms here :3
        self.norma_user_feat = nn.LayerNorm(embedding_dim)  # normalize content user branch
        self.norma_item_feat = nn.LayerNorm(embedding_dim)  # normalize content item branch
        self.norma_user_gmf = nn.LayerNorm(embedding_dim)  # normalize gmf user id branch
        self.norma_item_gmf = nn.LayerNorm(embedding_dim)  # normalize gmf item id branch
        self.norma_user_mlp = nn.LayerNorm(embedding_dim)  # normalize mlp user id branch
        self.norma_item_mlp = nn.LayerNorm(embedding_dim)  # normalize mlp item id branch
        # done :)
        ##### MY CODE ENDS #####

        # Combined MLP
        mlp_input_dim = 4 * embedding_dim
        mlp_layers = []
        prev_dim = mlp_input_dim
        for hidden_dim in hidden_dims:
            mlp_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        self.mlp = nn.Sequential(*mlp_layers)

        # Final Prediction Layer
        final_input_dim = embedding_dim + hidden_dims[-1]

        ##### Critical Issue Number: S5 #####
        # --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
        # self.final_layer = nn.Sequential(
        #     nn.Linear(final_input_dim, 32),
        #     nn.ReLU(),
        #     nn.Linear(32, 1),
        #     nn.Sigmoid()  # Output in [0, 1], scale to [1, 5]
        # )
        ##### MY CODE HERE #####
        """
        Sigmoid saturates near 0 and 1 which means ratings of 1 or 5 produce
        almost zero gradient and the model basically stops learning from them.
        Hardtanh is piecewise linear and has gradient of 1 for everything in
        the (-1, 1) range, so no saturation happens during normal training.
        I also adjust the scaling formula in the forward pass accordingly.
        """
        # starting my part :D
        self.predictor_capa = nn.Sequential(
            nn.Linear(final_input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Hardtanh(min_val=-1.0, max_val=1.0)  # constant gradient, no saturation at extremes
        )
        # done :p
        ##### MY CODE ENDS #####

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=0.01)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, user_id, item_id):
        # Get ID Embeddings
        user_emb_gmf = self.user_embedding_gmf(user_id)
        item_emb_gmf = self.item_embedding_gmf(item_id)
        user_emb_mlp = self.user_embedding_mlp(user_id)
        item_emb_mlp = self.item_embedding_mlp(item_id)

        # Get Content Features
        user_feat = self.user_features[user_id]
        item_feat = self.item_features[item_id]

        ##### Critical Issue Number: S9 (forward) #####
        # --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
        # user_feat_emb = self.user_feat_net(user_feat)
        # item_feat_emb = self.item_feat_net(item_feat)
        # gmf_user = user_emb_gmf + user_feat_emb
        # gmf_item = item_emb_gmf + item_feat_emb
        # mlp_input = torch.cat([user_emb_mlp, item_emb_mlp, user_feat_emb, item_feat_emb], dim=-1)
        ##### MY CODE HERE #####
        """
        Apply layer norms to each branch before fusion so they all have comparable
        scale before being added or concatenated together.
        """
        # my fix here <3
        user_feat_emb = self.norma_user_feat(self.user_feat_net(user_feat))  # normalize content
        item_feat_emb = self.norma_item_feat(self.item_feat_net(item_feat))  # normalize content
        user_emb_gmf_n = self.norma_user_gmf(user_emb_gmf)  # normalize gmf user id
        item_emb_gmf_n = self.norma_item_gmf(item_emb_gmf)  # normalize gmf item id
        user_emb_mlp_n = self.norma_user_mlp(user_emb_mlp)  # normalize mlp user id
        item_emb_mlp_n = self.norma_item_mlp(item_emb_mlp)  # normalize mlp item id
        gmf_user = user_emb_gmf_n + user_feat_emb  # fuse after normalization
        gmf_item = item_emb_gmf_n + item_feat_emb  # fuse after normalization
        mlp_input = torch.cat([user_emb_mlp_n, item_emb_mlp_n, user_feat_emb, item_feat_emb], dim=-1)
        # done here :3
        ##### MY CODE ENDS #####

        # GMF: Element-wise product (ID embeddings + feature embeddings)
        gmf_output = gmf_user * gmf_item

        # MLP
        mlp_output = self.mlp(mlp_input)

        # Final Prediction
        combined = torch.cat([gmf_output, mlp_output], dim=-1)

        ##### Critical Issue Number: S5 (forward) #####
        # --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
        # output = self.final_layer(combined)
        # rating = output.squeeze() * 4 + 1
        ##### MY CODE HERE #####
        """
        Use predictor_capa and adjust the scale. Hardtanh output of -1 maps to
        rating 1 and output of 1 maps to rating 5, so rating = output * 2 + 3.
        """
        # my scaling fix here :)
        salida_raw = self.predictor_capa(combined)  # hardtanh output in [-1, 1]
        rating = salida_raw.squeeze() * 2 + 3  # map [-1, 1] to [1, 5]
        # done :D
        ##### MY CODE ENDS #####

        return rating


## Step 6: Create DataLoaders

In [ ]:
class MovieLensDataset(Dataset):
    """Dataset for MovieLens ratings."""

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        return {
            'user_id': torch.tensor(row['user_id'], dtype=torch.long),
            'movie_id': torch.tensor(row['movie_id'], dtype=torch.long),
            'rating': torch.tensor(row['rating'], dtype=torch.float)
        }


batch_size = 512

##### Critical Issue Number: S6 #####
# --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
# train_dataset = MovieLensDataset(train_data)
# test_dataset = MovieLensDataset(filtered_test_data)
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=batch_size)
# print(f"Training batches: {len(train_loader)}")
# print(f"Test batches: {len(test_loader)}")
##### MY CODE HERE #####
"""
Here I carve a validation set from the training data so that early stopping
has a proper signal and test set is never touched during the training loop.
I use the last 10% of each user training interactions for validation, keeping
the temporal order correct inside the training window.
"""
# starting the split :3

train_data_final, val_data = chronological_split(train_data, train_ratio=0.9)  # 90/10 split from training

train_dataset = MovieLensDataset(train_data_final)  # actual training data
val_dataset = MovieLensDataset(val_data)  # validation, for early stopping only
test_dataset = MovieLensDataset(filtered_test_data)  # final evaluation, never used in training

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# done creating loaders :D
##### MY CODE ENDS #####


## Step 7: Training with Early Stopping

In [ ]:
# Model parameters
embedding_dim = 32
num_users = len(user_encoder.classes_)
num_movies = len(movie_encoder.classes_)

# Instantiate model
model = FeatureEnhancedNCF(
    num_users=num_users,
    num_items=num_movies,
    embedding_dim=embedding_dim,
    user_features=user_features,
    item_features=item_features,
    hidden_dims=[128, 64]
)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training configuration
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Early stopping parameters
best_loss = float('inf')
patience = 3
patience_counter = 0
num_epochs = 20

# Training loop
for epoch in range(num_epochs):
    # Training phase
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        user_id = batch['user_id']
        movie_id = batch['movie_id']
        ratings = batch['rating']

        optimizer.zero_grad()
        outputs = model(user_id, movie_id)
        loss = criterion(outputs, ratings)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}')

    ##### Critical Issue Number: S6 #####
    # --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
    # model.eval()
    # with torch.no_grad():
    #     eval_loss = 0
    #     for batch in test_loader:  # original uses test_loader here, which is the leakage
    #         user_id = batch['user_id']
    #         movie_id = batch['movie_id']
    #         ratings = batch['rating']
    #         outputs = model(user_id, movie_id)
    #         loss = criterion(outputs, ratings)
    #         eval_loss += loss.item()
    # avg_eval_loss = eval_loss / len(test_loader)
    # print(f'Test Loss: {avg_eval_loss:.4f}')
    ##### MY CODE HERE #####
    """
    Now early stopping monitors validation loss instead of test loss. This way
    the test set is never used during training and the final metrics are not
    inflated by the model selection process.
    """
    # using val_loader here :3
    model.eval()
    with torch.no_grad():
        perdida_val = 0  # accumulate validation loss
        for batch in val_loader:  # val set, not test
            user_id = batch['user_id']
            movie_id = batch['movie_id']
            ratings = batch['rating']
            outputs = model(user_id, movie_id)
            loss = criterion(outputs, ratings)
            perdida_val += loss.item()

    avg_eval_loss = perdida_val / len(val_loader)  # average over val batches
    print(f'Val Loss: {avg_eval_loss:.4f}')
    # done replacing test with val :)
    ##### MY CODE ENDS #####

    # Early stopping check
    if avg_eval_loss < best_loss:
        best_loss = avg_eval_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  New best model saved!')
    else:
        patience_counter += 1
        print(f'  No improvement. Patience: {patience_counter}/{patience}')
        if patience_counter >= patience:
            print(f'\nEarly stopping at epoch {epoch + 1}!')
            break

# Load best model
model.load_state_dict(torch.load('best_model.pt'))
print(f'\nLoaded best model with val loss: {best_loss:.4f}')

##### Critical Issue Number: S6 (final test eval) #####
# --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
# (original had no separate final test evaluation, test was used inside the loop)
##### MY CODE HERE #####
"""
After loading the best model, run one final evaluation on the test set. This is
the only time test data is used and ensure the metrics we report are truly on
unseen data.
"""
# final test eval here :D
model.eval()
with torch.no_grad():
    perdida_test = 0  # test loss accumulator
    for batch in test_loader:  # first and only time test set is evaluated
        user_id = batch['user_id']
        movie_id = batch['movie_id']
        ratings = batch['rating']
        outputs = model(user_id, movie_id)
        loss = criterion(outputs, ratings)
        perdida_test += loss.item()

avg_test_loss = perdida_test / len(test_loader)
print(f'Final Test Loss (one-shot, no leakage): {avg_test_loss:.4f}')
# that's it for training :3
##### MY CODE ENDS #####


## Step 8: Generate Predictions

In [ ]:
def generate_predictions(model, data_loader):
    """Generate predictions for all samples in the data loader."""
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in data_loader:
            user_id = batch['user_id']
            movie_id = batch['movie_id']
            outputs = model(user_id, movie_id)
            predictions.extend(zip(
                user_id.numpy(),
                movie_id.numpy(),
                outputs.numpy()
            ))
    return predictions


# Generate predictions
test_predictions = generate_predictions(model, test_loader)
print(f"Generated {len(test_predictions)} predictions")
print(f"Sample predictions: {test_predictions[:5]}")

## Step 9: Comprehensive Evaluation Metrics

In [ ]:
# =============================================================================
# RATING PREDICTION METRICS
# =============================================================================

def calculate_rmse(predictions, actuals):
    """Root Mean Square Error - measures rating prediction accuracy."""
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    return np.sqrt(np.mean((predictions - actuals) ** 2))


def calculate_mae(predictions, actuals):
    """Mean Absolute Error - measures average magnitude of errors."""
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    return np.mean(np.abs(predictions - actuals))


# =============================================================================
# RANKING METRICS
# =============================================================================

def dcg_at_k(relevance_scores, k):
    """Discounted Cumulative Gain at rank k."""
    relevance_scores = np.asarray(relevance_scores)[:k]
    if relevance_scores.size:
        return np.sum((2**relevance_scores - 1) / np.log2(np.arange(2, relevance_scores.size + 2)))
    return 0.0


def ndcg_at_k(relevance_scores, ideal_relevance_scores, k):
    """Normalized Discounted Cumulative Gain at rank k."""
    dcg = dcg_at_k(relevance_scores, k)
    idcg = dcg_at_k(ideal_relevance_scores, k)
    return dcg / idcg if idcg > 0 else 0.0


def precision_at_k(recommended_items, relevant_items, k):
    """Precision@K - proportion of recommended items that are relevant."""
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    hits = len(recommended_at_k & relevant_set)
    return hits / k if k > 0 else 0.0


def recall_at_k(recommended_items, relevant_items, k):
    """Recall@K - proportion of relevant items that are recommended."""
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    hits = len(recommended_at_k & relevant_set)
    return hits / len(relevant_set) if len(relevant_set) > 0 else 0.0


def hit_rate_at_k(recommended_items, relevant_items, k):
    """Hit Rate@K - whether any relevant item appears in top-K."""
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    return 1.0 if len(recommended_at_k & relevant_set) > 0 else 0.0


def reciprocal_rank(recommended_items, relevant_items):
    """Reciprocal Rank - inverse of the rank of the first relevant item."""
    relevant_set = set(relevant_items)
    for rank, item in enumerate(recommended_items, 1):
        if item in relevant_set:
            return 1.0 / rank
    return 0.0


def average_precision(recommended_items, relevant_items):
    """Average Precision - average of precision values at each relevant item position."""
    relevant_set = set(relevant_items)
    hits = 0
    sum_precision = 0.0
    for rank, item in enumerate(recommended_items, 1):
        if item in relevant_set:
            hits += 1
            sum_precision += hits / rank
    return sum_precision / len(relevant_set) if len(relevant_set) > 0 else 0.0


# =============================================================================
# COVERAGE METRICS
# =============================================================================

def catalog_coverage(all_recommendations, total_items):
    """Catalog Coverage - percentage of items recommended at least once."""
    unique_recommended = set(all_recommendations)
    return len(unique_recommended) / total_items if total_items > 0 else 0.0


def user_coverage(users_with_recommendations, total_users):
    """User Coverage - percentage of users who received recommendations."""
    return users_with_recommendations / total_users if total_users > 0 else 0.0

In [ ]:
class RecommenderMetrics:
    """Comprehensive evaluation class for recommender systems."""

    def __init__(self, relevance_threshold=4):
        """
        Parameters:
        - relevance_threshold: rating threshold to consider an item as relevant
                              (e.g., ratings >= 4 are considered relevant)
        """
        self.relevance_threshold = relevance_threshold

    def evaluate_all(self, predictions_df, test_data_df, k_values=[5, 10, 20],
                     num_items=None, num_users=None):
        """Compute all metrics for the recommender system."""

        # Merge predictions with actual ratings
        merged_df = pd.merge(
            test_data_df[['user_id', 'movie_id', 'rating']],
            predictions_df,
            on=['user_id', 'movie_id'],
            how='inner'
        )

        results = {}

        # Rating Prediction Metrics
        results['RMSE'] = calculate_rmse(
            merged_df['predicted_rating'].values,
            merged_df['rating'].values
        )
        results['MAE'] = calculate_mae(
            merged_df['predicted_rating'].values,
            merged_df['rating'].values
        )

        # Ranking Metrics (per user, then averaged)
        user_ids = merged_df['user_id'].unique()
        metrics_per_k = {k: {
            'ndcg': [], 'precision': [], 'recall': [],
            'hit_rate': [], 'mrr': [], 'map': []
        } for k in k_values}

        all_recommended_items = []
        users_with_recs = 0

        for user_id in user_ids:
            user_data = merged_df[merged_df['user_id'] == user_id].copy()
            if len(user_data) == 0:
                continue

            users_with_recs += 1
            user_data = user_data.sort_values('predicted_rating', ascending=False)

            recommended_items = user_data['movie_id'].tolist()
            actual_ratings = user_data['rating'].tolist()
            relevant_items = user_data[
                user_data['rating'] >= self.relevance_threshold
            ]['movie_id'].tolist()

            all_recommended_items.extend(recommended_items)

            for k in k_values:
                # NDCG@K
                relevance_scores = actual_ratings[:k]
                ideal_scores = sorted(actual_ratings, reverse=True)[:k]
                metrics_per_k[k]['ndcg'].append(ndcg_at_k(relevance_scores, ideal_scores, k))

                # Other metrics
                metrics_per_k[k]['precision'].append(precision_at_k(recommended_items, relevant_items, k))
                metrics_per_k[k]['recall'].append(recall_at_k(recommended_items, relevant_items, k))
                metrics_per_k[k]['hit_rate'].append(hit_rate_at_k(recommended_items, relevant_items, k))
                metrics_per_k[k]['mrr'].append(reciprocal_rank(recommended_items[:k], relevant_items))
                metrics_per_k[k]['map'].append(average_precision(recommended_items[:k], relevant_items))

        # Average metrics across users
        for k in k_values:
            results[f'NDCG@{k}'] = np.mean(metrics_per_k[k]['ndcg'])
            results[f'Precision@{k}'] = np.mean(metrics_per_k[k]['precision'])
            results[f'Recall@{k}'] = np.mean(metrics_per_k[k]['recall'])
            results[f'HitRate@{k}'] = np.mean(metrics_per_k[k]['hit_rate'])
            results[f'MRR@{k}'] = np.mean(metrics_per_k[k]['mrr'])
            results[f'MAP@{k}'] = np.mean(metrics_per_k[k]['map'])

        # Coverage Metrics
        if num_items is not None:
            results['CatalogCoverage'] = catalog_coverage(all_recommended_items, num_items)
        if num_users is not None:
            results['UserCoverage'] = user_coverage(users_with_recs, num_users)

        return results

    def print_results(self, results, k_values=[5, 10, 20]):
        """Pretty print the evaluation results."""
        print('=' * 60)
        print('RECOMMENDER SYSTEM EVALUATION RESULTS')
        print('=' * 60)

        print('\n📊 RATING PREDICTION METRICS')
        print('-' * 40)
        print(f"  RMSE:  {results['RMSE']:.4f}")
        print(f"  MAE:   {results['MAE']:.4f}")

        print('\n📈 RANKING METRICS')
        print('-' * 40)
        header = f"{'Metric':<15}" + "".join([f"@{k:<8}" for k in k_values])
        print(header)
        print('-' * len(header))

        for metric in ['NDCG', 'Precision', 'Recall', 'HitRate', 'MRR', 'MAP']:
            row = f"{metric:<15}"
            for k in k_values:
                value = results.get(f'{metric}@{k}', 0)
                row += f"{value:<9.4f}"
            print(row)

        if 'CatalogCoverage' in results or 'UserCoverage' in results:
            print('\n🎯 COVERAGE METRICS')
            print('-' * 40)
            if 'CatalogCoverage' in results:
                print(f"  Catalog Coverage: {results['CatalogCoverage']:.4f} "
                      f"({results['CatalogCoverage']*100:.2f}%)")
            if 'UserCoverage' in results:
                print(f"  User Coverage:    {results['UserCoverage']:.4f} "
                      f"({results['UserCoverage']*100:.2f}%)")

        print('\n' + '=' * 60)

## Step 10: Run Evaluation

In [ ]:
# Convert predictions to DataFrame
predictions_df = pd.DataFrame(
    test_predictions,
    columns=['user_id', 'movie_id', 'predicted_rating']
)

# Prepare test data DataFrame
test_data_df = filtered_test_data[['user_id', 'movie_id', 'rating']].copy()

# Initialize evaluator (items with rating >= 4 are considered relevant)
evaluator = RecommenderMetrics(relevance_threshold=4)

# Compute all metrics
results = evaluator.evaluate_all(
    predictions_df=predictions_df,
    test_data_df=test_data_df,
    k_values=[5, 10, 20],
    num_items=num_movies,
    num_users=num_users
)

# Print results
evaluator.print_results(results, k_values=[5, 10, 20])

##### Critical Issue Number: S10 #####
# --- ORIGINAL CODE (Keep commented out; DO NOT DELETE) ---
# (original evaluate_all ranks over only 3-4 test items per user, pool is too small)
##### MY CODE HERE #####
"""
The original metrics are inflated because for each user the ranking pool only
contains the items in their test partition, which is usually just 3 or 4 movies.
I sample 100 random items the user never interacted with as negatives and compute
NDCG and HitRate over this pool, which is a more honest evaluation. This follows
the protocol from Krichene and Rendle 2020 which showed that small pool metrics
are inconsistent estimators of full catalog performance.
"""
# starting pool evaluation :p

import random as _random

def evaluar_pool_muestreado(pred_df, test_df, train_df,
                             n_negativos=100, k_vals=[5, 10, 20], semilla=4222):
    '''
    Entry: prediction dataframe, test and train dataframes, number of negatives to sample
    Process: for each user pairs their relevant items with n_negativos random negatives,
             ranks by predicted rating and computes HitRate@K and NDCG@K over this pool
    Output: dict with HitRate@K and NDCG@K for each K value
    '''
    _random.seed(semilla)  # fix seed for reproducible results

    todos_los_items = set(pred_df['movie_id'].unique())  # full catalog visible to model

    historial_train = (train_df.groupby('user_id')['movie_id']
                       .apply(set).to_dict())  # map each user to their seen items in train

    acum = {k: {'hit': [], 'ndcg': []} for k in k_vals}  # store results per user

    for uid in test_df['user_id'].unique():
        usuario_test = test_df[test_df['user_id'] == uid]
        positivos = set(usuario_test[usuario_test['rating'] >= 4]['movie_id'].tolist())  # relevant items for this user

        if not positivos:
            continue  # skip if user has no relevant test items

        ya_vistos = historial_train.get(uid, set())  # items this user saw in training
        pool_candidatos = list(todos_los_items - ya_vistos - positivos)  # eligible negatives

        if len(pool_candidatos) < n_negativos:
            continue  # not enough candidates, skip

        negativos = _random.sample(pool_candidatos, n_negativos)  # sample the negatives
        pool_eval = list(positivos) + negativos  # evaluation pool for this user

        preds_uid = pred_df[pred_df['user_id'] == uid].set_index('movie_id')
        registros = [(iid,
                      float(preds_uid.loc[iid, 'predicted_rating']) if iid in preds_uid.index else 0.0,
                      iid in positivos)
                     for iid in pool_eval]

        registros.sort(key=lambda x: x[1], reverse=True)  # rank descending by predicted score
        items_rankeados = [r[0] for r in registros]

        for k in k_vals:
            topk = items_rankeados[:k]
            hit = 1.0 if any(i in positivos for i in topk) else 0.0
            relevancia = [1 if i in positivos else 0 for i in topk]
            ideal = sorted(relevancia, reverse=True)
            dcg_val = sum(r / np.log2(pos + 2) for pos, r in enumerate(relevancia))
            idcg_val = sum(r / np.log2(pos + 2) for pos, r in enumerate(ideal))
            ndcg_val = dcg_val / idcg_val if idcg_val > 0 else 0.0
            acum[k]['hit'].append(hit)
            acum[k]['ndcg'].append(ndcg_val)

    resultados = {}
    for k in k_vals:
        resultados[f'HitRate@{k} (pool-100-neg)'] = np.mean(acum[k]['hit'])
        resultados[f'NDCG@{k} (pool-100-neg)'] = np.mean(acum[k]['ndcg'])
    return resultados


print("\n--- S10 Fix: Evaluation over sampled negative pool (100 negatives per user) ---")
print("[Compare with the inflated metrics from evaluate_all above]")
resultados_pool = evaluar_pool_muestreado(
    predictions_df, filtered_test_data, train_data,
    n_negativos=100, k_vals=[5, 10, 20])

print("\nFair evaluation metrics (pool-corrected):")
for nombre, valor in resultados_pool.items():
    print(f"  {nombre}: {valor:.4f}")
print("\n[These should be lower than original metrics, showing the inflation from S10]")

# done with pool evaluation <3
##### MY CODE ENDS #####


---
## Use of AI

For this assignment I used an AI assistant (Claude) to help me organize and
express my ideas, to translate some parts from Spanish into English and to check
the grammar, since Spanish is my mother tongue and writing long technical
explanations in English is still challeging for me sometimes.

It also helped me to debug the PyTorch code — particularly the forward pass
changes for S5 and S9, where I had the right idea but I was not sure about the
exact syntax for `nn.Hardtanh` and `nn.LayerNorm`. Before writing any of the
solution schemes I revisited all the relevant professor's slides (Topics 3, 4,
and 6) and the He et al. NCF 2017 paper to make sure I had the theoretical
background. After that I used Claude to help me land the ideas into proper
English and to check that my code would not break the run-all requirement.

I also used it to help me understand the Krichenie & Rendle (2020) paper for S10,
because at first I misunderstood what "sampled metrics" means in they paper. Once
Claude explained the mechanism I re-wrote the solution scheme myself to make sure
I understood it correctly, and aswell the code for `evaluar_pool_muestreado`.
